<a href="https://colab.research.google.com/github/codenumber4x-ctrl/Agi/blob/ccr-51fce02c-klz683/large_language_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

this is the start we are to study and create llm models which are presented as a potential path towards human level general intelligence, we study and work, explore the math and code and do ablation tests

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as f
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
import time
import os
import pandas as pd
import random
import argparse
import torch.optim as optim
import pathlib
from pathlib import Path


we know the skeleton rather well, ofcourse everything starts with data so lets get some data, search the kaggle which is directly here or use \
import opendatasets as od \
of corse after the pip install\
 or simply use the kagglehub.dataset_download() but since this is colab we use simple function lets find some good data

In [6]:
import kagglehub
path = kagglehub.dataset_download("jjinho/wikipedia-20230701")

Using Colab cache for faster access to the 'wikipedia-20230701' dataset.


In [2]:
pwd

'/content'

In [3]:
data_path = Path(path)
data_path.resolve()

NameError: name 'Path' is not defined

In [ ]:
cd /root/.kagglehub/.datasets

[Errno 2] No such file or directory: '/root/.kagglehub/.datasets'
/root


In [ ]:
pwd

'/root'

was simply checking the paths

In [7]:
os.listdir(path) # to show the files in it

['x.parquet',
 'h.parquet',
 'w.parquet',
 'g.parquet',
 'a.parquet',
 'y.parquet',
 'l.parquet',
 'n.parquet',
 'i.parquet',
 'number.parquet',
 'j.parquet',
 'm.parquet',
 'b.parquet',
 'r.parquet',
 'v.parquet',
 'z.parquet',
 'o.parquet',
 'wiki_2023_index.parquet',
 'k.parquet',
 'q.parquet',
 'e.parquet',
 'f.parquet',
 'p.parquet',
 't.parquet',
 'other.parquet',
 'd.parquet',
 'u.parquet',
 's.parquet',
 'c.parquet']

In [8]:
data = pd.read_parquet("/root/.cache/kagglehub/datasets/jjinho/wikipedia-20230701/versions/2/a.parquet")
# so i have just copied the path, i think it failed initially bse i didnt pass the whole path
# and should have used os.path.join ### ....maybe havent tested

In [9]:
data.head(4)

,id,title,text,categories
0,49495844,A & B High Performance Firearms,A & B High Performance Firearms was a competit...,"[Defunct firearms manufacturers, Defunct manuf..."
1,3579086,A & C Black,A & C Black is a British book publishing compa...,"[Encyclopædia Britannica, Ornithological publi..."
2,62397582,A & F Harvey Brothers,"A & F Harvey Brothers, first Spinning Cotton M...",[Cotton mills]
3,15547032,A & G Price,A & G Price Limited is an engineering firm and...,"[Locomotive manufacturers of New Zealand, Tham..."


In [12]:
data.index

RangeIndex(start=0, stop=442726, step=1)

In [4]:
data.describe()

NameError: name 'data' is not defined

In [11]:
data.value_counts()

TypeError: unhashable type: 'numpy.ndarray'

 the above is frankly ineficient should have used print and f"" string(format string literal)


In [13]:
print(f"the shape is {data.shape}")
print(f"the columns are {data.columns}")
print(f"the null values are {data.isnull().sum()}")

the shape is (442726, 4)
the columns are Index(['id', 'title', 'text', 'categories'], dtype='object')
the null values are id            0
title         0
text          0
categories    0
dtype: int64


In [14]:
data.head(10)

,id,title,text,categories
0,49495844,A & B High Performance Firearms,A & B High Performance Firearms was a competit...,"[Defunct firearms manufacturers, Defunct manuf..."
1,3579086,A & C Black,A & C Black is a British book publishing compa...,"[Encyclopædia Britannica, Ornithological publi..."
2,62397582,A & F Harvey Brothers,"A & F Harvey Brothers, first Spinning Cotton M...",[Cotton mills]
3,15547032,A & G Price,A & G Price Limited is an engineering firm and...,"[Locomotive manufacturers of New Zealand, Tham..."
4,8021609,A & M Karagheusian,thumb|right|238px|A portion of the Karagheusia...,"[1904 establishments in the United States, Arm..."
5,28380942,A & P Food Stores Building,"The A & P Food Stores Building in St. Louis, M...","[Art Deco architecture in Missouri, Buildings ..."
6,38966582,A & R Recording,A & R Recording Inc. was a major American inde...,"[1958 establishments in New York City, 1958 in..."
7,20016047,A (1965 film),A is a 1965 short film animated by Jan Lenica....,"[1965 films, West German films, German animate..."
8,24604084,A (1998 Japanese film),A is a 1998 Japanese documentary film about th...,"[1998 films, Japanese documentary films, 1998 ..."
9,23488527,A (1998 Kannada film),A is an 1998 Indian Kannada-language romantic ...,"[Films set in Bangalore, 1998 films, 1990s Kan..."


In [21]:
text_data = data["text"]


In [34]:
d =list(text_data.head(1))

In [35]:
[list(i) for i in d]

[['A',
  ' ',
  '&',
  ' ',
  'B',
  ' ',
  'H',
  'i',
  'g',
  'h',
  ' ',
  'P',
  'e',
  'r',
  'f',
  'o',
  'r',
  'm',
  'a',
  'n',
  'c',
  'e',
  ' ',
  'F',
  'i',
  'r',
  'e',
  'a',
  'r',
  'm',
  's',
  ' ',
  'w',
  'a',
  's',
  ' ',
  'a',
  ' ',
  'c',
  'o',
  'm',
  'p',
  'e',
  't',
  'i',
  't',
  'i',
  'o',
  'n',
  ' ',
  'p',
  'i',
  's',
  't',
  'o',
  'l',
  ' ',
  'm',
  'a',
  'n',
  'u',
  'f',
  'a',
  'c',
  't',
  'u',
  'r',
  'e',
  'r',
  '.',
  ' ',
  'P',
  'r',
  'o',
  'd',
  'u',
  'c',
  't',
  's',
  ' ',
  'i',
  'n',
  'c',
  'l',
  'u',
  'd',
  'e',
  'd',
  ' ',
  't',
  'h',
  'e',
  ' ',
  '"',
  'L',
  'i',
  'm',
  'i',
  't',
  'e',
  'd',
  ' ',
  'C',
  'l',
  'a',
  's',
  's',
  '"',
  ' ',
  'a',
  'n',
  'd',
  ' ',
  '"',
  'O',
  'p',
  'e',
  'n',
  ' ',
  'C',
  'l',
  'a',
  's',
  's',
  '"',
  ' ',
  's',
  'e',
  'm',
  'i',
  '-',
  'a',
  'u',
  't',
  'o',
  'm',
  'a',
  't',
  'i',
  'c',
  ' ',
  'p',
  'i',